<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/11-Adding_Hybrid_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hybrid Search: Combining Vector and Keyword Retrieval

This notebook implements **hybrid search** by combining vector (semantic) retrieval with keyword-based retrieval using LlamaIndex. Hybrid search often outperforms either approach alone, especially for queries containing proper nouns, version numbers, or technical terms that semantic search may under-weight.

## Learning Objectives

By the end of this notebook, you will be able to:
- Load a pre-built vector store from HuggingFace Hub
- Build a `SimpleKeywordTableIndex` from the same document nodes
- Implement a custom `HybridRetriever` that interleaves vector and keyword results
- Evaluate hybrid vs. vector-only retrieval using `RetrieverEvaluator` (Hit Rate, MRR)

## Prerequisites

- Familiarity with LlamaIndex `VectorStoreIndex` and retrievers
- Basic understanding of RAG pipelines
- OpenAI API key (set in Colab Secrets as `OPENAI_API_KEY`)

## Install Packages and Setup Variables


In [ ]:
!pip install -q llama-index==0.14.0 openai==1.107.0 chromadb==1.0.21 llama-index-vector-stores-chroma==0.5.3 \
                llama-index-embeddings-huggingface==0.6.0 llama-index-finetuning==0.4.0 \
                llama-index-embeddings-openai==0.5.2 jedi==0.19.2

In [ ]:
import os

# Load the OpenAI API key from Colab Secrets, with a local environment variable fallback.
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
except ImportError:
    os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "")

In [ ]:
# Allows running asyncio in environments with an existing event loop, like Jupyter notebooks.

import nest_asyncio

nest_asyncio.apply()

# Initialize Models

In [ ]:
from llama_index.llms.openai import OpenAI
from llama_index.core import Settings
from llama_index.embeddings.openai import OpenAIEmbedding

Settings.llm = OpenAI(model="gpt-5-mini", additional_kwargs={'reasoning_effort':'minimal'})
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

# Download knowledge base


In [ ]:
from huggingface_hub import hf_hub_download

hf_hub_download(repo_id="jaiganesan/ai_tutor_knowledge", filename="vectorstore.zip", repo_type="dataset", local_dir=".")

In [ ]:
!unzip -o vectorstore.zip

# Create Vector Index from Persisted Chroma Store

In [ ]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import VectorStoreIndex

# Load the vector store from the local storage.
db = chromadb.PersistentClient(path="./ai_tutor_knowledge")
chroma_collection = db.get_collection("ai_tutor_knowledge")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

# Create the index based on the vector store.
vector_index = VectorStoreIndex.from_vector_store(vector_store=vector_store)

# Create Keyword Index from the Same Document Nodes

In [ ]:
def retrieve_all_nodes_from_vector_index(vector_index, query="Whatever", similarity_top_k=100000000):
    # Set similarity_top_k to a large number to retrieve all the nodes
    vector_retriever = vector_index.as_retriever(similarity_top_k=similarity_top_k)

    # Retrieve all nodes
    all_nodes = vector_retriever.retrieve(query)
    nodes = [item.node for item in all_nodes]

    return nodes

nodes = retrieve_all_nodes_from_vector_index(vector_index)
print(len(nodes))

In [ ]:
from llama_index.core import SimpleKeywordTableIndex

# Build the keyword table index from all the document nodes.
keyword_index = SimpleKeywordTableIndex(nodes=nodes)

## Optional: What Does the Keyword Index Store?

In [ ]:
# Inspect what keywords are indexed in the SimpleKeywordTableIndex
try:
    keyword_table = keyword_index.index_struct
    all_keywords = list(keyword_table.table.keys())
    print(f"Total unique keywords indexed: {len(all_keywords)}")
    print(f"\nSample keywords (first 30):")
    print(", ".join(sorted(all_keywords)[:30]))
    print(f"\nThese keywords are matched exactly against the query,")
    print(f"complementing the semantic matching from the vector index.")
except Exception as e:
    print(f"Could not inspect keyword index: {e}")

# Hybrid Retriever


In [ ]:
from llama_index.core import QueryBundle
from llama_index.core.schema import NodeWithScore
from llama_index.core.retrievers import (
    BaseRetriever,
    VectorIndexRetriever,
    KeywordTableSimpleRetriever,
)
from typing import List

class HybridRetriever(BaseRetriever):
    """Hybrid retriever that performs both semantic search and keyword search."""

    def __init__(
        self,
        vector_retriever: VectorIndexRetriever,
        keyword_retriever: KeywordTableSimpleRetriever,
        max_retrieve: int = 10,
    ) -> None:
        """Init params."""

        self._vector_retriever = vector_retriever
        self._keyword_retriever = keyword_retriever
        self._max_retrieve = max_retrieve
        super().__init__()

    def _retrieve(self, query_bundle: QueryBundle) -> List[NodeWithScore]:
        """Retrieve nodes given query."""

        vector_nodes = self._vector_retriever.retrieve(query_bundle)
        keyword_nodes = self._keyword_retriever.retrieve(query_bundle)

        resulting_nodes = []
        node_ids_added = set()
        for i in range(min(len(vector_nodes), len(keyword_nodes))):
            vector_node = vector_nodes[i]
            if vector_node.node.node_id not in node_ids_added:
                resulting_nodes += [vector_node]
                node_ids_added.add(vector_node.node.node_id)

            keyword_node = keyword_nodes[i]
            if keyword_node.node.node_id not in node_ids_added:
                resulting_nodes += [keyword_node]
                node_ids_added.add(keyword_node.node.node_id)

        return resulting_nodes

# Test hybrid retriever vs vector retriever

In [ ]:
from llama_index.core import get_response_synthesizer
from llama_index.core.query_engine import RetrieverQueryEngine

# Create hybrid query engine
vector_retriever = VectorIndexRetriever(index=vector_index, similarity_top_k=6)
keyword_retriever = KeywordTableSimpleRetriever(index=keyword_index, num_chunks_per_query=6)
hybrid_retriever = HybridRetriever(vector_retriever, keyword_retriever, max_retrieve=6)
response_synthesizer = get_response_synthesizer(llm=Settings.llm)
hybrid_query_engine = RetrieverQueryEngine(
    retriever=hybrid_retriever,
    response_synthesizer=response_synthesizer,
)

# Test the query engine
answer = hybrid_query_engine.query("How does KOSMOS-2 work?")
print(answer)

In [ ]:
# Create vector query engine
vector_retriever = VectorIndexRetriever(index=vector_index, similarity_top_k=6)
vector_query_engine = RetrieverQueryEngine(
    retriever=vector_retriever,
    response_synthesizer=response_synthesizer,
)

# Test the query engine
answer = vector_query_engine.query("How does KOSMOS-2 work?")
print(answer)

## Optional: Hybrid vs Vector-Only — Side-by-Side

In [ ]:
# Run the same query through both retrievers and compare what each finds
test_query = "What safety measures does LLaMA 2 have?"
query_bundle = QueryBundle(test_query)

print(f"Query: {test_query}\n")
print("=" * 60)

# Vector-only
try:
    vector_nodes = vector_retriever.retrieve(test_query)
    print(f"Vector-only ({len(vector_nodes)} nodes):")
    for i, n in enumerate(vector_nodes[:3], 1):
        print(f"  [{i}] score={n.score:.4f if n.score else 'N/A'} | {n.text[:100]}...")
except Exception as e:
    print(f"Vector retriever error: {e}")

print()

# Hybrid
try:
    hybrid_nodes = hybrid_retriever.retrieve(test_query)
    print(f"Hybrid ({len(hybrid_nodes)} nodes):")
    for i, n in enumerate(hybrid_nodes[:3], 1):
        print(f"  [{i}] score={n.score:.4f if n.score else 'N/A'} | {n.text[:100]}...")
except Exception as e:
    print(f"Hybrid retriever error: {e}")

print(f"\nHybrid typically adds keyword-matched chunks that pure vector search misses,")
print(f"especially for proper nouns, version numbers, and technical terms.")

# Evaluate

Run the following code if you want to generate an evaluation dataset from scratch. You can choose to download an evaluation dataset running the cell after this one.

In [ ]:
# from llama_index.core.evaluation import generate_question_context_pairs

# # Create questions for each segment. These questions will be used to
# # assess whether the retriever can accurately identify and return the
# # corresponding segment when queried.
# rag_eval_dataset = generate_question_context_pairs(
#     nodes, llm=Settings.llm, num_questions_per_chunk=1
# )

# # We can save the evaluation dataset as a json file for later use.
# rag_eval_dataset.save_json("./rag_eval_dataset_question_context.json")

You can download a version of the evaluation dataset with the following code cell, so that you don't have to create the eval dataset from scratch with the code above.

In [ ]:
from huggingface_hub import hf_hub_download
from llama_index.finetuning.embeddings.common import EmbeddingQAFinetuneDataset

# Download the evaluation dataset
hf_hub_download(repo_id="jaiganesan/ai_tutor_knowledge", filename="rag_eval_dataset_question_context_subset_50.json", repo_type="dataset", local_dir=".")
rag_eval_dataset = EmbeddingQAFinetuneDataset.from_json("./rag_eval_dataset_question_context_subset_50.json")

In [ ]:
import asyncio
import pandas as pd
from llama_index.core.evaluation import RetrieverEvaluator


def from_eval_results_to_dataframe(name, eval_results):
    """Format retriever evaluation results as a summary string."""
    metric_dicts = [r.metric_vals_dict for r in eval_results]
    full_df = pd.DataFrame(metric_dicts)
    hit_rate = full_df["hit_rate"].mean()
    mrr = full_df["mrr"].mean()
    return f"{name}: Hit Rate = {hit_rate:.4f}, MRR = {mrr:.4f}"


# Evaluate retrievers at different top_k values.
loop = asyncio.get_event_loop()
for i in [2, 4, 6, 8, 10]:
    # Evaluate hybrid retriever
    vector_retriever = VectorIndexRetriever(index=vector_index, similarity_top_k=i)
    keyword_retriever = KeywordTableSimpleRetriever(index=keyword_index, num_chunks_per_query=i)
    hybrid_retriever = HybridRetriever(vector_retriever, keyword_retriever, max_retrieve=i)
    retriever_evaluator = RetrieverEvaluator.from_metric_names(
        ["mrr", "hit_rate"], retriever=hybrid_retriever
    )
    eval_results = loop.run_until_complete(
        retriever_evaluator.aevaluate_dataset(rag_eval_dataset)
    )
    print(from_eval_results_to_dataframe(f"Hybrid retriever top_{i}", eval_results))

    # Evaluate vector retriever
    vector_retriever_k = VectorIndexRetriever(index=vector_index, similarity_top_k=i)
    retriever_evaluator = RetrieverEvaluator.from_metric_names(
        ["mrr", "hit_rate"], retriever=vector_retriever_k
    )
    eval_results = loop.run_until_complete(
        retriever_evaluator.aevaluate_dataset(rag_eval_dataset)
    )
    print(from_eval_results_to_dataframe(f"Vector retriever top_{i}", eval_results))